In [2]:
import transformers, jlens

import json
import itertools
from collections import defaultdict
 
import torch
from transformer_lens import HookedTransformer

In [11]:
LAYERS_TO_PROBE = list(range(0, 12, 2))

def load_dataset(path):
    with open(path) as f:
        data = json.load(f)
    groups = defaultdict(list)
    for row in data["prompts"]:
        groups[(row["cluster_id"], row["condition"])].append(row)
    return data["clusters"], groups


def find_word_token_pos(model, prompt: str, word: str) -> int:
    full_toks = model.to_str_tokens(prompt)
    target = word.lower()
    for i in range(len(full_toks) - 1, -1, -1):
        if full_toks[i].strip().lower() == target:
            return i
    raise ValueError(f"Could not locate token for '{word}' in: {full_toks}")


def single_prompt_jacobian(model,
                           prompt: str,
                           layer: int,
                           source_pos: int) -> torch.Tensor:
    """Exact J_{l,t,t'} for one prompt: d_model x d_model, transposed to act
    from the right (h @ J), t' fixed at the final sequence position."""
    tokens = model.to_tokens(prompt)
    final_pos = tokens.shape[1] - 1
    resid_name = f"blocks.{layer}.hook_resid_post"
    final_resid_name = f"blocks.{model.cfg.n_layers - 1}.hook_resid_post"
 
    _, cache = model.run_with_cache(tokens, names_filter=resid_name)
    h0 = cache[resid_name][0, source_pos].detach().clone()  # (d_model,)
 
    def f(h):
        captured = {}

        def patch_hook(resid, hook):
            # write h into position `source_pos` while preserving grad flow through h
            resid = resid.clone()
            resid[0, source_pos] = h
            return resid

        def capture_hook(resid, hook):
            captured["final"] = resid
            return resid

        model.run_with_hooks(
            tokens,
            fwd_hooks=[
                (resid_name, patch_hook),
                (final_resid_name, capture_hook),
            ],
            return_type=None,
        )
        return captured["final"][0, final_pos]

    J = torch.autograd.functional.jacobian(f, h0, vectorize=True)  # (d_model_out, d_model_in)
    return J.T

In [4]:
def group_averaged_jacobian(model, prompts_words, layer: int) -> torch.Tensor:
    """Average J_{l,t,t'} over every prompt in a (cluster, condition) group."""
    mats = []
    for row in prompts_words:
        pos = find_word_token_pos(model, row["prompt"], row["word"])
        mats.append(single_prompt_jacobian(model, row["prompt"], layer, pos))
    return torch.stack(mats).mean(dim=0)


In [ ]:
def compare(J_A: torch.Tensor, J_B: torch.Tensor) -> dict:
    diff = J_A - J_B
    fro_A = J_A.norm().item()
    fro_diff = diff.norm().item()
    cos = torch.nn.functional.cosine_similarity(
        J_A.flatten().unsqueeze(0), J_B.flatten().unsqueeze(0)
    ).item()
    return dict(frobenius=fro_diff, frobenius_relative=fro_diff / (fro_A + 1e-9), cosine=cos)

In [6]:
def lens_topk(model, h_probe: torch.Tensor, J_l: torch.Tensor, k=10):
    """softmax(norm(h_probe @ J_l) @ W_U), top-k token strings."""
    h = (h_probe @ J_l).unsqueeze(0).unsqueeze(0)
    h = model.ln_final(h)
    logits = h @ model.W_U + model.b_U
    top = logits[0, 0].topk(k).indices.tolist()
    return [model.to_single_str_token(t) for t in top]

In [7]:
model = HookedTransformer.from_pretrained("gpt2")
model.eval()
torch.set_grad_enabled(True)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2 into HookedTransformer


torch.autograd.grad_mode.set_grad_enabled(mode=True)

In [8]:
clusters, groups = load_dataset("synonym_jlens_dataset.json")

In [12]:
results = []
for cluster in clusters:
    cid = cluster["cluster_id"]
    conditions = ["base", "synonym", "control"]
    for layer in LAYERS_TO_PROBE:
        J = {}
        for cond in conditions:
            J[cond] = group_averaged_jacobian(model, groups[(cid, cond)], layer)

        base_vs_syn = compare(J["base"], J["synonym"])
        base_vs_ctrl = compare(J["base"], J["control"])

        results.append(dict(
            cluster=cid, layer=layer,
            base_vs_synonym=base_vs_syn,
            base_vs_control=base_vs_ctrl,
            invariance_gap=base_vs_ctrl["frobenius_relative"] - base_vs_syn["frobenius_relative"],
        ))
        print(f"[{cid}] layer {layer:2d}  "
                f"base~syn rel-Frob={base_vs_syn['frobenius_relative']:.3f} cos={base_vs_syn['cosine']:.3f}  |  "
                f"base~ctrl rel-Frob={base_vs_ctrl['frobenius_relative']:.3f} cos={base_vs_ctrl['cosine']:.3f}")


[size] layer  0  base~syn rel-Frob=0.810 cos=0.629  |  base~ctrl rel-Frob=1.314 cos=0.357
[size] layer  2  base~syn rel-Frob=0.644 cos=0.768  |  base~ctrl rel-Frob=1.031 cos=0.488
[size] layer  4  base~syn rel-Frob=0.567 cos=0.826  |  base~ctrl rel-Frob=0.858 cos=0.627
[size] layer  6  base~syn rel-Frob=0.457 cos=0.891  |  base~ctrl rel-Frob=0.717 cos=0.741
[size] layer  8  base~syn rel-Frob=0.332 cos=0.945  |  base~ctrl rel-Frob=0.529 cos=0.860
[size] layer 10  base~syn rel-Frob=0.177 cos=0.985  |  base~ctrl rel-Frob=0.268 cos=0.965
[speed] layer  0  base~syn rel-Frob=0.832 cos=0.680  |  base~ctrl rel-Frob=1.158 cos=0.344
[speed] layer  2  base~syn rel-Frob=0.632 cos=0.816  |  base~ctrl rel-Frob=0.946 cos=0.495
[speed] layer  4  base~syn rel-Frob=0.486 cos=0.886  |  base~ctrl rel-Frob=0.872 cos=0.588
[speed] layer  6  base~syn rel-Frob=0.382 cos=0.930  |  base~ctrl rel-Frob=0.756 cos=0.711
[speed] layer  8  base~syn rel-Frob=0.263 cos=0.966  |  base~ctrl rel-Frob=0.533 cos=0.858
[spee

In [13]:
with open("group_jacobian_comparison.json", "w") as f:
        json.dump(results, f, indent=2)
print("\nSaved results -> group_jacobian_comparison.json")



Saved results -> group_jacobian_comparison.json


# ----------------------------

In [ ]:
hf = transformers.AutoModelForCausalLM.from_pretrained("openai-community/gpt2").cuda()
tok = transformers.AutoTokenizer.from_pretrained("openai-community/gpt2")
model = jlens.from_hf(hf, tok)

In [5]:
model

HFLensModel(GPT2LMHeadModel, n_layers=12, d_model=768)

In [7]:
import json
items = json.load(open("../datasets/gender_test_rephrased_v2.json"))
prompts = [it["rephrased_context"].replace("BLANK", it["targets"]["stereotype"])
           for it in items]

In [9]:
from jlens.examples import load_wikitext_prompts
prompts = load_wikitext_prompts(n_prompts=200)   # ~100 is usable, paper uses 1000

In [10]:
lens = jlens.fit(
    model,
    prompts,
    dim_batch=32,
    max_seq_len=128,
    checkpoint_path="out/gpt2_ckpt.pt",
    checkpoint_every=10,
)

In [11]:
prompt = "My father is a very"
lens_logits, model_logits, input_ids = lens.apply(
    model, prompt, positions=[-1]
)

In [13]:
for layer, logits in sorted(lens_logits.items()):
    print(layer, [tok.decode([t]) for t in logits[0].topk(5).indices])

0 ['ModLoader', ' tremend', ' !!', ' cryst', '********************************']
1 [' enthusi', ' VERY', ' !!', ' tremend', ' FANT']
2 [' VERY', ' enthusi', ' negro', ' FANT', ' brill']
3 [' VERY', ' alot', ' enthusi', ' FANT', ' brill']
4 [' enthusi', ' surv', ' alot', ' VERY', ' FANT']
5 [' enthusi', ' VERY', ' very', ' alot', ' tremend']
6 [' VERY', ' very', ' nice', ' wonderful', 'very']
7 [' nice', ' VERY', ' wonderful', ' very', ' dear']
8 [' talented', ' fortunate', ' enthusi', ' welf', ' dear']
9 [' talented', ' passionate', ' enthusi', ' compassionate', ' generous']
10 [' talented', ' devout', ' conscientious', ' nice', ' generous']
